# KNN 
Assignment 2 

CS 4267

## Notes 
- This was written in Visual Studio Code. 
- This was written and tested with Python 3.12.5 version. 
- The requirements.txt file contains all necessary dependencies. 
- Once dependencies are installed and the csv file is in the correct location, this should run correctly when "Run All" is selected. 

## Should the data be normalized? 

After reviewing the data, it was apparent that the values were not normalized. Some values were very large while others were very small. When using KNN classification, if one attribute has large values and another has small values, the attribute with large values will dominate the distance comparison. Due to this, it is necessary to normalize the data for this dataset before using KNN classification. There are several different ways to normalize data. I chose to use min max normalization. This decision was due to there being some very large and some very small values in the dataset. Min max normalization normalizes all values to be within the same range. This put all attributes on the same scale and prevented larger valued attributes from dominating the distance measurements. 

## Imports

In [15]:
# Imports needed
import pandas as pd
import numpy as np 
from sklearn.preprocessing import MinMaxScaler

## Distance calculations

In [16]:
# Calculate the Euclidean distance between the training samples and testing samples 
def calculate_distances(train_arr, test_sample): 
    distances = []

    # Exclude the label from the attributes 
    test_attributes = test_sample[:-1]

    for train_sample in train_arr: 
        # Separate the label and attributes 
        train_label = train_sample[-1]
        train_attributes = train_sample[:-1]

        # Calculate the Euclidean distance between the test and train attributes 
        dist = np.linalg.norm(test_attributes - train_attributes)

        # Store the distance and training label 
        dist_info = (dist, train_label)

        # Append the distance and training label info to the distances list 
        distances.append(dist_info)

    # Return the list of distances and training labels 
    return distances

## Sample assignment/classification

In [17]:
# Split the data into training and testing using a given training percentage (default: 70%) and seed (default: 36)
def split_data(df, train_percentage=0.7, seed=36): 
    # Randomly sample the training df 
    train_df = df.sample(frac=train_percentage, random_state=seed)

    # Drop the training df indices to create the testing df 
    test_df = df.drop(train_df.index)
    
    # Return the train and test dfs 
    return train_df, test_df

In [18]:
# Finds the k minimum samples from the given list of distance info 
def find_min_samples(k, dist_info): 
    # Turn the list into a df 
    dist_info_df = pd.DataFrame(dist_info)

    # Sort the values into a list by their distances 
    sorted_dist_info = dist_info_df.sort_values(by=0).values.tolist()

    # Get the first k (lowest distance) samples 
    min_samples = sorted_dist_info[:k]
    
    # Return the minimum samples 
    return min_samples

In [19]:
# Determine the classification of a sample given the minimum distance info from training samples 
def classify_sample(min_dist_info): 
    positive_count = 0
    negative_count = 0

    # Increment the positive and negative count based on the minimum training sample classifications 
    for info in min_dist_info: 
        if info[1] > 0: 
            positive_count += 1 
        else: 
            negative_count += 1

    # Return 1 if more of the closest training samples were positive, 0 otherwise 
    if positive_count > negative_count: 
        return 1 
    else: 
        return 0 

In [20]:
# Assign the samples to a class 
def knn_classification(df, k): 
    results = []

    # Set the initial true positive, true negative, false negative, and false positive counts to 0 
    tp_count = 0
    tn_count = 0
    fn_count = 0
    fp_count = 0

    # Split the data for training and testing 
    train_df, test_df = split_data(df)

    # Convert the dataframes to numpy arrays 
    train_arr = train_df.to_numpy()
    test_arr = test_df.to_numpy()

    # Iterate through each test sample in the test array 
    for test_sample in test_arr: 
        # Get the actual class of the test sample 
        actual_class = test_sample[-1]

        # Calculate the euclidean distances between the test sample and each sample in the training data 
        dist_info = calculate_distances(train_arr, test_sample)

        # Find the k closest training samples to the test sample 
        min_samples = find_min_samples(k, dist_info)

        # Predict the class of the sample based on the closest training samples 
        pred_class = classify_sample(min_samples)

        # Increment the count metric 
        if actual_class > 0: 
            if pred_class > 0: 
                tp_count += 1
            else: 
                fn_count += 1
        else: 
            if pred_class > 0: 
                fp_count += 1
            else: 
                tn_count += 1

        # Append the predicted class to the results list 
        results.append(pred_class)

    # Calculate the accuracy 
    tot_tp_tn = tp_count + tn_count 
    tot_tp_tn_fp_fn = tp_count + tn_count + fp_count + fn_count 
    accuracy = tot_tp_tn / tot_tp_tn_fp_fn

    # Create the confusion matrix 
    confusion_matrix = np.array([[tp_count, fn_count],
                                 [fp_count, tn_count]])

    # Return the results list, accuracy, and confusion matrix 
    return results, accuracy, confusion_matrix

## Read in the data

In [ ]:
# Read in the dataset 
dataset_path = "knn_dataset.csv"
df = pd.read_csv(dataset_path, header=None)

# Display the dataset
df

,0,1,2,3,4,5,6,7,8,9,...,21,22,23,24,25,26,27,28,29,30
0,8.196,16.84,51.71,201.9,0.08600,0.05943,0.015880,0.005917,0.1769,0.06503,...,21.96,57.26,242.2,0.12970,0.13570,0.06880,0.02564,0.3105,0.07409,-1
1,13.170,18.66,85.98,534.6,0.11580,0.12310,0.122600,0.073400,0.2128,0.06777,...,27.95,102.80,759.4,0.17860,0.41660,0.50060,0.20880,0.3900,0.11790,1
2,12.050,14.63,78.04,449.3,0.10310,0.09092,0.065920,0.027490,0.1675,0.06043,...,20.70,89.88,582.6,0.14940,0.21560,0.30500,0.06548,0.2747,0.08301,-1
3,13.490,22.30,86.91,561.0,0.08752,0.07698,0.047510,0.033840,0.1809,0.05718,...,31.82,99.00,698.8,0.11620,0.17110,0.22820,0.12820,0.2871,0.06917,-1
4,11.760,21.60,74.72,427.9,0.08637,0.04966,0.016570,0.011150,0.1495,0.05888,...,25.72,82.98,516.5,0.10850,0.08615,0.05523,0.03715,0.2433,0.06563,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
608,11.220,33.81,70.79,386.8,0.07780,0.03574,0.004967,0.006434,0.1845,0.05828,...,41.78,78.44,470.9,0.09994,0.06885,0.02318,0.03002,0.2911,0.07307,-1
609,20.510,27.81,134.40,1319.0,0.09159,0.10740,0.155400,0.083400,0.1448,0.05592,...,37.38,162.70,1872.0,0.12230,0.27610,0.41460,0.15630,0.2437,0.08328,1
610,9.567,15.91,60.21,279.6,0.08464,0.04087,0.016520,0.016670,0.1551,0.06403,...,19.16,65.74,335.9,0.15040,0.09515,0.07161,0.07222,0.2757,0.08178,-1
611,14.030,21.25,89.79,603.4,0.09070,0.06945,0.014620,0.018960,0.1517,0.05835,...,30.28,98.27,715.5,0.12870,0.15130,0.06231,0.07963,0.2226,0.07617,-1


## Normalize the data

In [22]:
# Normalizes the data using Min Max Scalar normalization 
def normalize(df): 
    # Use MinMaxScalar() from sklearn 
    scaler = MinMaxScaler()
    normalized_array = scaler.fit_transform(df)

    # Turn the normalized array into a dataframe 
    normalized_df = pd.DataFrame(normalized_array)
    
    # Return the normalized dataframe 
    return normalized_df

In [23]:
# Normalize and display the data 
normalized_df = normalize(df)

normalized_df

,0,1,2,3,4,5,6,7,8,9,...,21,22,23,24,25,26,27,28,29,30
0,0.057504,0.241123,0.054730,0.024772,0.301255,0.122845,0.037207,0.029409,0.358081,0.317397,...,0.264925,0.034115,0.014009,0.386515,0.105180,0.054952,0.088110,0.303568,0.124951,0.0
1,0.292915,0.302672,0.291549,0.165896,0.570281,0.318140,0.287254,0.364811,0.539394,0.375105,...,0.424574,0.260919,0.141123,0.709437,0.377711,0.399840,0.717526,0.460280,0.412305,1.0
2,0.239907,0.166385,0.236680,0.129714,0.455629,0.219434,0.154452,0.136630,0.310606,0.220514,...,0.231343,0.196574,0.097670,0.516608,0.182699,0.243610,0.225017,0.232998,0.183458,0.0
3,0.308060,0.425769,0.297975,0.177094,0.314977,0.176676,0.111317,0.168191,0.378283,0.152064,...,0.527719,0.241994,0.126229,0.297365,0.139525,0.182268,0.440550,0.257441,0.092680,0.0
4,0.226182,0.402097,0.213738,0.120636,0.304595,0.092878,0.038824,0.055417,0.219697,0.187869,...,0.365139,0.162209,0.081424,0.246517,0.057106,0.044113,0.127663,0.171102,0.069461,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
608,0.200625,0.815015,0.186580,0.103203,0.227228,0.050181,0.011638,0.031978,0.396465,0.175232,...,0.793177,0.139599,0.070217,0.189989,0.040322,0.018514,0.103162,0.265326,0.118261,0.0
609,0.640305,0.612107,0.626149,0.498621,0.351720,0.269983,0.364105,0.414513,0.195960,0.125527,...,0.675906,0.559241,0.414569,0.337648,0.241397,0.331150,0.537113,0.171890,0.185229,1.0
610,0.122391,0.209672,0.113468,0.057731,0.288977,0.065916,0.038707,0.082853,0.247980,0.296335,...,0.190299,0.076348,0.037038,0.523212,0.065838,0.057196,0.248179,0.234969,0.175390,0.0
611,0.333617,0.390260,0.317877,0.195080,0.343685,0.153580,0.034255,0.094235,0.230808,0.176706,...,0.486674,0.238358,0.130333,0.379912,0.120315,0.049768,0.273643,0.130298,0.138594,0.0


## Test using different k values  

### k = 1

In [24]:
# Test using k=1

results_1, accuracy_1, confusion_matrix_1 = knn_classification(normalized_df, 1)
confusion_matrix_df_1 = pd.DataFrame(confusion_matrix_1, columns=['+', '-'], index=['+', '-'])

print(f"Results: {results_1}\n")
print(f"Accuracy: {accuracy_1}")

confusion_matrix_df_1

Results: [0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0]

Accuracy: 0.967391304347826


,+,-
+,65,3
-,3,113


### k = 3

In [25]:
# Test using k=3

results_3, accuracy_3, confusion_matrix_3 = knn_classification(normalized_df, 3)
confusion_matrix_df_3 = pd.DataFrame(confusion_matrix_3, columns=['+', '-'], index=['+', '-'])

print(f"Results: {results_3}\n")
print(f"Accuracy: {accuracy_3}")

confusion_matrix_df_3

Results: [0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0]

Accuracy: 0.9945652173913043


,+,-
+,67,1
-,0,116


### k = 5

In [26]:
# Test using k=5

results_5, accuracy_5, confusion_matrix_5 = knn_classification(normalized_df, 5)
confusion_matrix_df_5 = pd.DataFrame(confusion_matrix_5, columns=['+', '-'], index=['+', '-'])

print(f"Results: {results_5}\n")
print(f"Accuracy: {accuracy_5}")

confusion_matrix_df_5

Results: [0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0]

Accuracy: 0.9945652173913043


,+,-
+,67,1
-,0,116


### k = 7

In [27]:
# Test using k=7

results_7, accuracy_7, confusion_matrix_7 = knn_classification(normalized_df, 7)
confusion_matrix_df_7 = pd.DataFrame(confusion_matrix_7, columns=['+', '-'], index=['+', '-'])

print(f"Results: {results_7}\n")
print(f"Accuracy: {accuracy_7}")
confusion_matrix_df_7

Results: [0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0]

Accuracy: 0.9836956521739131


,+,-
+,65,3
-,0,116


### k = 9

In [28]:
# Test using k=9

results_9, accuracy_9, confusion_matrix_9 = knn_classification(normalized_df, 9)
confusion_matrix_df_9 = pd.DataFrame(confusion_matrix_9, columns=['+', '-'], index=['+', '-'])

print(f"Results: {results_9}\n")
print(f"Accuracy: {accuracy_9}")

confusion_matrix_df_9

Results: [0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0]

Accuracy: 0.9782608695652174


,+,-
+,64,4
-,0,116
